# Complex CNN với Parallel Block

Notebook này tự đọc dữ liệu, tạo DataLoader, xây dựng model và huấn luyện. Code được viết từng bước để người mới dễ theo dõi.

In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import precision_recall_fscore_support

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
# Tìm thư mục gốc của project.
current_dir = Path.cwd().resolve()
possible_dirs = [
    current_dir,
    current_dir / "plant-disease-classification",
    current_dir.parent,
]

PROJECT_DIR = None

for folder in possible_dirs:
    has_readme = (folder / "README.md").is_file()
    has_notebooks = (folder / "notebooks").is_dir()

    if has_readme and has_notebooks:
        PROJECT_DIR = folder
        break

if PROJECT_DIR is None:
    raise FileNotFoundError("Không tìm thấy thư mục project.")

print("Project:", PROJECT_DIR)

Project: D:\Plant_disease\plant-disease-classification


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Notebook này yêu cầu GPU NVIDIA có CUDA.
if not torch.cuda.is_available():
    raise RuntimeError("Không tìm thấy GPU CUDA.")

torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device("cuda")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [4]:
RESULTS_DIR = PROJECT_DIR / "outputs" / "results"
MODELS_DIR = PROJECT_DIR / "outputs" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_CSV_PATH = RESULTS_DIR / "processed_data.csv"
CLASS_NAMES_PATH = RESULTS_DIR / "class_names.json"

if not PROCESSED_CSV_PATH.is_file():
    raise FileNotFoundError(
        "Không tìm thấy processed_data.csv. "
        "Hãy chạy 02_preprocessing.ipynb trước."
    )

processed_df = pd.read_csv(PROCESSED_CSV_PATH)

with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as file:
    class_names = json.load(file)

required_columns = {"processed_path", "class_name", "class_id", "split"}
missing_columns = required_columns - set(processed_df.columns)

if missing_columns:
    raise ValueError(
        f"processed_data.csv thiếu cột: {sorted(missing_columns)}"
    )

num_classes = len(class_names)
class_to_id = {name: index for index, name in enumerate(class_names)}
expected_ids = processed_df["class_name"].map(class_to_id)

if expected_ids.isna().any() or not np.array_equal(
    processed_df["class_id"].to_numpy(),
    expected_ids.to_numpy(),
):
    raise ValueError("class_id không khớp class_names.json.")

IMAGE_SIZE = 224
# Batch 64 đã được kiểm tra trên RTX 3050 6GB.
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001

print("Number of classes:", num_classes)
print("Processed manifest:", PROCESSED_CSV_PATH)
print(processed_df["split"].value_counts())

Number of classes: 38
Processed manifest: D:\Plant_disease\plant-disease-classification\outputs\results\processed_data.csv
split
train         37984
test          10873
validation     5427
Name: count, dtype: int64


## Tạo Dataset và DataLoader

CPU mở file ảnh và resize để tạo batch. Sau đó batch được chuyển lên GPU để augmentation và huấn luyện.

In [5]:
class PlantDiseaseDataset(Dataset):
    def __init__(self, dataframe, project_dir, transform):
        self.paths = dataframe["processed_path"].astype(str).tolist()
        self.labels = dataframe["class_id"].astype(int).tolist()
        self.project_dir = Path(project_dir)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        relative_path = self.paths[index].replace("\\", "/")
        image_path = self.project_dir.joinpath(*relative_path.split("/"))
        label = self.labels[index]

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        return image, label

In [6]:
# Ảnh đã được preprocessing thành RGB 224x224.
# CPU chỉ mở JPEG và chuyển ảnh thành tensor.
image_transform = transforms.ToTensor()

train_df = processed_df[processed_df["split"] == "train"]
val_df = processed_df[processed_df["split"] == "validation"]
test_df = processed_df[processed_df["split"] == "test"]

train_dataset = PlantDiseaseDataset(train_df, PROJECT_DIR, image_transform)
val_dataset = PlantDiseaseDataset(val_df, PROJECT_DIR, image_transform)
test_dataset = PlantDiseaseDataset(test_df, PROJECT_DIR, image_transform)

# Kiểm tra ảnh preprocessing có thật sự tồn tại.
first_image_path = PROJECT_DIR.joinpath(
    *train_dataset.paths[0].replace("\\", "/").split("/")
)

if not first_image_path.is_file():
    raise FileNotFoundError(
        "Không tìm thấy ảnh đã preprocessing. "
        "Hãy chạy lại 02_preprocessing.ipynb."
    )

# num_workers=0 giúp code ổn định và dễ chạy trên Windows.
# pin_memory giúp chuyển batch từ RAM lên GPU nhanh hơn.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 594
Validation batches: 85
Test batches: 170


In [7]:
# Các phép augmentation này nhận tensor CUDA và trả về tensor CUDA.
gpu_train_augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, scale=(0.90, 1.10)),
    transforms.ColorJitter(contrast=0.10),
])

def augment_images_on_gpu(images):
    return gpu_train_augmentation(images)

## Xây dựng Complex CNN

Parallel Block đưa cùng một đầu vào qua ba nhánh. Mỗi nhánh dùng một kernel khác nhau, sau đó kết quả được ghép lại.

In [8]:
class ParallelBlock(nn.Module):
    def __init__(self, input_channels):
        super().__init__()

        # Nhánh 1: convolution 1x1.
        self.branch1 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        # Nhánh 2: convolution 3x3.
        self.branch2 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        # Nhánh 3: convolution 5x5.
        self.branch3 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

    def forward(self, x):
        output1 = self.branch1(x)
        output2 = self.branch2(x)
        output3 = self.branch3(x)

        # 3 nhánh x 32 kênh = 96 kênh.
        output = torch.cat(
            [output1, output2, output3],
            dim=1,
        )

        return output

In [9]:
class ComplexCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.parallel_block = ParallelBlock(input_channels=64)

        self.block3 = nn.Sequential(
            nn.Conv2d(96, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.average_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.5)
        self.output_layer = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.parallel_block(x)
        x = self.block3(x)

        x = self.average_pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.dropout(x)
        x = self.output_layer(x)

        return x

In [10]:
model = ComplexCNN(num_classes=num_classes)
model = model.to(device)

total_parameters = 0

for parameter in model.parameters():
    total_parameters += parameter.numel()

# Kiểm tra model bằng một batch giả nằm trên GPU.
sample_images = torch.zeros(
    2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device
)

with torch.no_grad():
    sample_outputs = model(sample_images)

print(model)
print("Model device:", next(model.parameters()).device)
print("Output shape:", sample_outputs.shape)
print(f"Total parameters: {total_parameters:,}")

ComplexCNN(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (parallel_block): ParallelBlock(
    (branch1): Sequential(
      (0): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (branch2): Sequential(
      (0): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True

In [11]:
loss_function = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

# Mixed precision giúp GPU RTX train nhanh hơn và dùng ít VRAM hơn.
scaler = torch.amp.GradScaler("cuda")

MODEL_PATH = MODELS_DIR / "complex_cnn_best.pt"
HISTORY_PATH = RESULTS_DIR / "complex_cnn_history.json"
METRICS_PATH = RESULTS_DIR / "complex_cnn_metrics.json"

In [12]:
def train_one_epoch(model, dataloader):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_images = 0

    for images, labels in dataloader:
        # Chuyển batch từ RAM lên GPU.
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Augmentation được thực hiện trên GPU.
        images = augment_images_on_gpu(images)

        optimizer.zero_grad()

        # Forward và loss dùng mixed precision trên GPU.
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images)
            loss = loss_function(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        predictions = outputs.argmax(dim=1)
        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_images += batch_size

    average_loss = total_loss / total_images
    accuracy = total_correct / total_images

    return average_loss, accuracy

In [13]:
def evaluate(model, dataloader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_images = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda"):
                outputs = model(images)
                loss = loss_function(outputs, labels)
            predictions = outputs.argmax(dim=1)
            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (predictions == labels).sum().item()
            total_images += batch_size

    average_loss = total_loss / total_images
    accuracy = total_correct / total_images

    return average_loss, accuracy

In [14]:
best_val_loss = float("inf")
best_epoch = 0

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

for epoch in range(EPOCHS):
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader
    )

    val_loss, val_accuracy = evaluate(
        model, val_loader
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    # Chỉ lưu model khi validation loss tốt hơn.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_names": class_names,
                "image_size": IMAGE_SIZE,
            },
            MODEL_PATH,
        )

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

with open(HISTORY_PATH, "w", encoding="utf-8") as file:
    json.dump(history, file, indent=2)

print("Best epoch:", best_epoch)
print("Saved model:", MODEL_PATH)
print("Saved history:", HISTORY_PATH)

Epoch [1/10] | Train Loss: 1.7216 | Train Acc: 0.5355 | Val Loss: 0.9915 | Val Acc: 0.7089
Epoch [2/10] | Train Loss: 1.0495 | Train Acc: 0.6908 | Val Loss: 0.7466 | Val Acc: 0.7875
Epoch [3/10] | Train Loss: 0.8310 | Train Acc: 0.7528 | Val Loss: 0.6400 | Val Acc: 0.7955
Epoch [4/10] | Train Loss: 0.6906 | Train Acc: 0.7904 | Val Loss: 0.5230 | Val Acc: 0.8323
Epoch [5/10] | Train Loss: 0.6099 | Train Acc: 0.8138 | Val Loss: 0.3974 | Val Acc: 0.8789
Epoch [6/10] | Train Loss: 0.5428 | Train Acc: 0.8339 | Val Loss: 0.3727 | Val Acc: 0.8869
Epoch [7/10] | Train Loss: 0.4940 | Train Acc: 0.8468 | Val Loss: 0.5534 | Val Acc: 0.8233
Epoch [8/10] | Train Loss: 0.4386 | Train Acc: 0.8651 | Val Loss: 0.5708 | Val Acc: 0.8189
Epoch [9/10] | Train Loss: 0.4177 | Train Acc: 0.8706 | Val Loss: 0.2241 | Val Acc: 0.9370
Epoch [10/10] | Train Loss: 0.3920 | Train Acc: 0.8789 | Val Loss: 0.2517 | Val Acc: 0.9228
Best epoch: 9
Saved model: D:\Plant_disease\plant-disease-classification\outputs\models\c

In [15]:
# Nạp lại model tốt nhất trước khi test.
checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_accuracy = evaluate(model, test_loader)

all_predictions = []
all_labels = []
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.tolist())

precision, recall, f1_score, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0,
)

metrics = {
    "best_epoch": best_epoch,
    "test_loss": test_loss,
    "accuracy": test_accuracy,
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1_score),
}

with open(METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print(f"Test Loss : {test_loss:.4f}")
print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1_score:.4f}")
print("Saved metrics:", METRICS_PATH)

Test Loss : 0.2261
Accuracy  : 0.9328
Precision : 0.9356
Recall    : 0.9327
F1-score  : 0.9326
Saved metrics: D:\Plant_disease\plant-disease-classification\outputs\results\complex_cnn_metrics.json
